# Rezolvitor de probleme de informatica cu Gemini

**Inainte de rulare:** in bara din stanga apasa pe iconita cu cheie (**Secrets**), adauga un secret cu numele `GEMINI_API_KEY`, lipeste cheia ta din Google AI Studio si activeaza **Notebook access**.

Apoi ruleaza celulele in ordine.

In [ ]:
!pip install -q -U google-genai

In [ ]:
from google import genai
from google.genai import types
from google.colab import userdata
from IPython.display import display, Markdown

client = genai.Client(api_key=userdata.get('GEMINI_API_KEY'))
MODEL = "gemini-3.5-flash-lite"

# Test rapid ca cheia functioneaza
print(client.models.generate_content(model=MODEL, contents="Spune doar: OK").text)

OK


In [ ]:
INSTRUCTIUNI = """Esti un profesor de informatica foarte bun, care pregateste elevi pentru bacalaureat si olimpiada.
Primesti enuntul unei probleme si raspunzi in limba romana, structurat EXACT asa:

## 1. Intelegerea problemei
Ce se cere, ce date avem, ce restrictii conteaza.

## 2. Ideea de rezolvare
Algoritmul ales si DE CE l-ai ales (mentioneaza pe scurt si o abordare mai naiva, daca exista).

## 3. Codul complet
Cod corect, complet, compilabil, cu comentarii scurte.

## 4. Explicatie pas cu pas
Explica fiecare parte importanta a codului.

## 5. Exemplu de rulare
Ia un exemplu concret si arata cum evolueaza variabilele pas cu pas.

## 6. Complexitate
Complexitatea in timp si in memorie, cu justificare.
"""

def rezolva_problema(enunt, limbaj="C++"):
    prompt = f"Limbaj de programare: {limbaj}\n\nEnuntul problemei:\n{enunt.strip()}"
    config = types.GenerateContentConfig(system_instruction=INSTRUCTIUNI, temperature=0.2)

    text = ""
    afisare = display(Markdown("*Se gandeste...*"), display_id=True)
    for bucata in client.models.generate_content_stream(model=MODEL, contents=prompt, config=config):
        if bucata.text:
            text += bucata.text
            afisare.update(Markdown(text))
    return text

In [ ]:
enunt = """
Se citeste un numar natural n (1 <= n <= 100000) si apoi n numere intregi.
Sa se determine suma maxima a unei secvente de elemente consecutive din sir.

Exemplu:
Intrare: 8
         -2 1 -3 4 -1 2 1 -5
Iesire:  6
"""

raspuns = rezolva_problema(enunt, limbaj="C++")

## 1. Intelegerea problemei
Se dă un șir de $n$ numere întregi (care pot fi atât pozitive, cât și negative). Se cere să determinăm suma maximă pe care o poate avea o secvență de elemente *consecutive* din acest șir. 
*   **Ce avem:** $n$ (numărul de elemente) și cele $n$ numere ale șirului.
*   **Ce se cere:** Suma maximă a unei subsecvențe (Maximum Subarray Sum).
*   **Restricții:** $1 \le n \le 100.000$, ceea ce înseamnă că o soluție ineficientă ($O(n^2)$ sau $O(n^3)$) va primi TLE (Time Limit Exceeded). Avem nevoie de o soluție liniară $O(n)$.

## 2. Ideea de rezolvare
O abordare naivă ar fi să verificăm toate perechile posibile de capete ale secvenței (start și sfârșit) și să calculăm suma pentru fiecare. Aceasta ar duce la o complexitate de $O(n^2)$ sau chiar $O(n^3)$, mult prea lentă pentru $n = 100.000$.

Pentru a rezolva problema în timp **$O(n)$**, folosim **Algoritmul lui Kadane**. 
Ideea de bază este să parcurgem șirul o singură dată și să menținem două valori:
1.  `suma_curenta`: suma secvenței care se termină în elementul curent.
2.  `suma_maxima`: cea mai mare sumă găsită până în momentul curent.

La fiecare pas (când adăugăm un nou element $x$), avem de luat o decizie: fie extindem secvența anterioară adăugând $x$, fie începem o secvență nouă chiar de la $x$ (dacă suma anterioară a devenit negativă și ne trage în jos). Matematic: `suma_curenta = max(x, suma_curenta + x)`. Apoi actualizăm `suma_maxima = max(suma_maxima, suma_curenta)`.

## 3. Codul complet
```cpp
#include <iostream>
#include <algorithm>

using namespace std;

int main() {
    // Optimizare pentru citire/afisare rapida
    ios_base::sync_with_stdio(false);
    cin.tie(NULL);

    int n;
    if (!(cin >> n)) return 0;

    int x;
    // Citim primul element pentru a initializa sumele
    cin >> x;
    long long suma_curenta = x;
    long long suma_maxima = x;

    // Citim restul de n-1 elemente si aplicam Kadane
    for (int i = 2; i <= n; ++i) {
        cin >> x;
        // Decidem: extindem secventa veche sau incepem una noua de la x
        suma_curenta = max((long long)x, suma_curenta + x);
        // Actualizam maximul global
        suma_maxima = max(suma_maxima, suma_curenta);
    }

    cout << suma_maxima << "\n";

    return 0;
}
```

## 4. Explicatie pas cu pas
1.  **Optimizarea I/O:** `ios_base::sync_with_stdio(false); cin.tie(NULL);` este esențială la problemele cu mult date de intrare (cum e cazul aici, $N = 100.000$).
2.  **Tipurile de date:** Folosim `long long` pentru sume (`suma_curenta` și `suma_maxima`) deoarece elementele pot fi negative sau pozitive, dar suma maximă s-ar putea acumula și depăși limitele unui `int` clasic pe 32 de biți (deși în cazul datelor de la Bacalaureat sau probleme tipice s-ar putea să încapă, e o măsură bună de precauție).
3.  **Inițializarea:** Citim primul element și setăm atât `suma_curenta` cât și `suma_maxima` la valoarea lui $x$. Acest lucru ne asigură că algoritmul funcționează corect chiar și atunci când *toate* elementele din șir sunt negative.
4.  **Bucla principală (`for`):** De la al doilea element încolo, pentru fiecare număr `x` citit, aplicăm formula lui Kadane: `suma_curenta = max((long long)x, suma_curenta + x);`.
5.  **Actualizarea maximului:** Comparăm `suma_curenta` cu `suma_maxima` și reținem valoarea cea mai mare în `suma_maxima`.

## 5. Exemplu de rulare
Pentru șirul din enunț: `n = 8`, elementele: `-2 1 -3 4 -1 2 1 -5`

*   **Inițializare:** `x = -2`, `suma_curenta = -2`, `suma_maxima = -2`
*   **Pasul 1 (`i = 2`, `x = 1`):**
    *   `suma_curenta = max(1, -2 + 1) = max(1, -1) = 1` (am renunțat la `-2` și am început o secvență nouă de la `1`)
    *   `suma_maxima = max(-2, 1) = 1`
*   **Pasul 2 (`i = 3`, `x = -3`):**
    *   `suma_curenta = max(-3, 1 + (-3)) = max(-3, -2) = -2`
    *   `suma_maxima = max(1, -2) = 1`
*   **Pasul 3 (`i = 4`, `x = 4`):**
    *   `suma_curenta = max(4, -2 + 4) = max(4, 2) = 4` (începem o secvență nouă de la `4`)
    *   `suma_maxima = max(1, 4) = 4`
*   **Pasul 4 (`i = 5`, `x = -1`):**
    *   `suma_curenta = max(-1, 4 + (-1)) = max(-1, 3) = 3`
    *   `suma_maxima = max(4, 3) = 4`
*   **Pasul 5 (`i = 6`, `x = 2`):**
    *   `suma_curenta = max(2, 3 + 2) = max(2, 5) = 5` (secvența devine `[4, -1, 2]`, suma = 5)
    *   `suma_maxima = max(4, 5) = 5`
*   **Pasul 6 (`i = 7`, `x = 1`):**
    *   `suma_curenta = max(1, 5 + 1) = max(1, 6) = 6` (secvența devine `[4, -1, 2, 1]`, suma = 6)
    *   `suma_maxima = max(5, 6) = 6`
*   **Pasul 7 (`i = 8`, `x = -5`):**
    *   `suma_curenta = max(-5, 6 + (-5)) = max(-5, 1) = 1`
    *   `suma_maxima = max(6, 1) = 6`

Ieșire finală: `6` (secvența este `[4, -1, 2, 1]`).

## 6. Complexitate
*   **Timp:** $O(n)$. Parcurgem șirul o singură dată printr-o buclă care execută operații elementare de timp $O(1)$ pentru fiecare din cele $n$ elemente. Este optim.
*   **Memorie (Spațiu):** $O(1)$. Nu salvăm șirul într-un tablou (`vector` sau `array`), ci citim fiecare număr „din zbor” (on-the-fly) și păstrăm doar variabilele necesare pentru calcul. Consumul de memorie este minim, excelent pentru limite stricte de memorie la olimpiade.

**Pentru o problema noua:** schimba textul din `enunt` (si optional `limbaj="Python"`) si ruleaza din nou ultima celula de cod.